In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch.optim import AdamW, lr_scheduler
from torch.amp import autocast, GradScaler
import math
import numpy as np
import torch.multiprocessing as mp
mp.set_start_method("fork", force=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cuda


In [68]:
class RotaryPositionalEmbeddings(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, offset=0):
        B, N, H, D = x.shape
        t = torch.arange(offset, offset + N, device=x.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos()[None, :, None, :]   # 1, N, 1, D
        sin = emb.sin()[None, :, None, :]
        return x * cos + self._rotate_half(x) * sin

    def _rotate_half(self, x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

class TimestepEmbedding(nn.Module):
    def __init__(self, d_model, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(
            nn.Linear(freq_dim, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )
 
    def sinusoidal(self, t):
        half = self.freq_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / half
        )
        args = t[:, None].float() * freqs[None, :] * 1000.0
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return emb
 
    def forward(self, t):
        return self.mlp(self.sinusoidal(t))
    
class AdaLN(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.lin = nn.Linear(d_model, 6*d_model)
        nn.init.zeros_(self.lin.weight)
        nn.init.zeros_(self.lin.bias)

    def forward(self, t_emb):
        out = self.lin(F.silu(t_emb))
        return out.chunk(6, dim = -1)


class MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.2):
        assert d_model % num_heads == 0
        super().__init__()

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.num_heads = num_heads
        self.d_head = int(d_model / num_heads)
        self.scale = self.d_head ** -0.5

        self.rope = RotaryPositionalEmbeddings(self.d_head)

    def forward(self, x, attn_mask = None):
        B, N, D = x.shape
        QKV = self.Wqkv(x)
        QKV = QKV.view(B, N, 3, self.num_heads, self.d_head)
        QKV = QKV.permute(2, 0, 3, 1, 4)
        Q, K, V = QKV[0], QKV[1], QKV[2] # B H N D

        Q = self.rope(Q.permute(0, 2, 1, 3)).permute(0, 2, 1, 3)
        K = self.rope(K.permute(0, 2, 1, 3)).permute(0, 2, 1, 3)

        score = Q @ K.transpose(-2, -1) # batch H N S
        score = score * self.scale

        if attn_mask is not None:
            score = score.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))

        weights = F.softmax(score, dim = -1)
        out = torch.matmul(weights, V) # batch H N d_head

        out = out.transpose(1, 2).contiguous() # B N H d_head
        out = out.view(B, N, -1)
        return self.Wo(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, heads, dropout = 0.2):
        super().__init__()
        self.attn = MHA(d_model, heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.adaln = AdaLN(d_model)
        self.dropout = nn.Dropout(p=dropout)

    
    def modulate(self, x, shift, scale):
        # x: (B, N, D), shift/scale: (B, D)
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


    def forward(self, x, t_emb, attn_mask = None):
        a_shift, a_scale, a_gate, ff_shift, ff_scale, ff_gate = self.adaln(t_emb)

        attended = self.attn(self.modulate(self.ln1(x), a_shift, a_scale), attn_mask = attn_mask)
        x = x + a_gate[:, None, :] * self.dropout(attended)

        fed = self.ff(self.modulate(self.ln2(x), ff_shift, ff_scale))
        x = x + ff_gate[:, None, :] * self.dropout(fed)
        return x

class FlowMatcher(nn.Module):
    def __init__(self, vocab_size, d_model, d_time, num_heads, d_ff, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.t_embed = TimestepEmbedding(d_model, d_time)
        self.transformers = nn.ModuleList(
            [TransformerBlock(d_model, d_ff, num_heads) for i in range(num_layers)]
        )
        self.ln = nn.LayerNorm(d_model)
        self.adaln = AdaLN(d_model)

        self.unembed = nn.Linear(d_model, vocab_size, bias = False)
        self.unembed.weight = self.embed.weight

    def forward(self, x, t, attn_mask = None):
        out = self.embed(x)
        t_emb = self.t_embed(t)

        for trans in self.transformers:
            out = trans(out, t_emb, attn_mask=attn_mask)

        shift, scale, *_ = self.adaln(t_emb)
        norm = self.ln(out)
        pred = self.unembed((1 + scale[:, None, :]) * norm + shift[:, None, :])

        return pred


In [69]:
class TextHolder(Dataset):
    def __init__(self, text, max_seq_len = 128):
        super().__init__()

        chars = sorted(set(text))
        self.vocab_size = len(chars)
        self.max_seq_len = max_seq_len

        self.stoi = {c : i for i, c in enumerate(chars)}
        self.itos = {i: c for i, c in enumerate(chars)} 
        self.encode = lambda s: [self.stoi[c] for c in s]
        self.decode = lambda l: ''.join([self.itos[i] for i in l])

        self.tokens = self.encode(text)
        self.num_samples = int(len(self.tokens) / max_seq_len)

    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, index):
        return self.tokens[index * self.max_seq_len : (index + 1) * self.max_seq_len]

def sample(x1, t, vocab_size):    
    B, N = x1.shape
    keep_prob = t.view(B, 1)
    keep_mask = torch.rand(B, N, device=x1.device) < keep_prob
    random_tokens = torch.randint(0, vocab_size, (B, N), device=x1.device)
    x_t = torch.where(keep_mask, x1, random_tokens)
    return x_t, keep_mask

def collate(batch, vocab_size, min_step):
    x1 = torch.vstack([torch.tensor(t) for t in batch])
    t = torch.rand(len(batch)).clamp(min_step, 1.0 - min_step)
    xt, _ = sample(x1, t, vocab_size)
    return ({"x" : xt, "t" : t}, x1)

file = open("data/tinyshakespeare.txt", "r").read()
train_split = int(len(file) * 0.8)

train_dataset = TextHolder(file[:train_split])
val_dataset = TextHolder(file[train_split:])
vocab_size = train_dataset.vocab_size

def collate_wrapper(batch):
    return collate(batch, vocab_size, 0.1)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_wrapper)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_wrapper)

In [70]:
def loss_fn(y_hat, x1, t, attn_mask = None):
    B, N, V = y_hat.shape
    # ce = F.cross_entropy(y_hat.transpose(1, 2), x1, reduction="none")
    # time_weighting = ((1.0 - t) / t).view(B, 1)

    # return (ce * time_weighting).sum() / float(B * N)

    return F.cross_entropy(y_hat.view(-1, V), x1.view(-1))

In [ ]:
epochs = 500
warmup_steps = 50
lr_full = 3e-4

whereswaldo = FlowMatcher(
    vocab_size=train_dataset.vocab_size,
    d_model=128, 
    d_time=256, 
    num_heads=4, 
    d_ff=512, 
    num_layers=8
).to(device)

print(f"Parameters: {sum(p.numel() for p in whereswaldo.parameters())}")
print(f"embed params:      {sum(p.numel() for n, p in whereswaldo.named_parameters() if 'embed' in n):,}")
print(f"transformer params:{sum(p.numel() for n, p in whereswaldo.named_parameters() if 'transformer' in n):,}")

optimizer = AdamW(whereswaldo.parameters(), lr=lr_full, weight_decay=0.1)
def lr_lambda(step):
    if step < warmup_steps: return step / max(1, warmup_steps)
    return 1
scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda)
nn.init.normal_(whereswaldo.embed.weight, std=0.02)

Parameters: 1941120
embed params:      57,728
transformer params:1,784,064


Parameter containing:
tensor([[-0.0143,  0.0123, -0.0156,  ..., -0.0175, -0.0090,  0.0523],
        [-0.0259, -0.0294, -0.0245,  ...,  0.0139, -0.0288, -0.0225],
        [ 0.0121,  0.0132, -0.0288,  ..., -0.0244, -0.0069, -0.0035],
        ...,
        [-0.0506,  0.0149,  0.0331,  ...,  0.0040,  0.0004, -0.0107],
        [-0.0061,  0.0213, -0.0163,  ..., -0.0236,  0.0355, -0.0202],
        [-0.0330,  0.0054, -0.0001,  ..., -0.0033,  0.0229,  0.0003]],
       device='cuda:0', requires_grad=True)

In [76]:
LOG_STEP = 20
EMA = math.log(vocab_size)
pbar = tqdm(range(epochs))

for i in pbar:
    whereswaldo.train()
    for batch, labels in train_loader:
        optimizer.zero_grad()
        batch = {k : v.to(device) for k, v in batch.items()}
        labels = labels.to(device)
        logits = whereswaldo(**batch)

        loss = loss_fn(logits, labels, batch["t"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(whereswaldo.parameters(), max_norm=1.0)
        optimizer.step()

        EMA = 0.99 * EMA + 0.01 * loss.item()
        pbar.set_postfix(loss=EMA)

    scheduler.step()

100%|██████████| 200/200 [15:06<00:00,  4.53s/it, loss=1.52]


In [ ]:
@torch.no_grad()
def generate(model, seq_len, vocab_size, num_steps=64, temperature=1.0,
             confidence_threshold=0.5, device="cuda", batch_size=1):
    """
    Uniform-state discrete diffusion sampler. Starts from a canvas of
    random tokens (no [MASK] token needed, since your model was trained
    with uniform-state corruption, not masking) and iteratively refines
    it, re-noising any position the model isn't confident about.
    """
    model.eval()

    # start: every position is a random token -- there's no absorbing
    # "noise" state to initialize to, unlike masking
    x_t = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    eps = 1e-2
    # t goes from "mostly noise" up to "clean" -- same direction as training's
    # kappa(t)=t convention, where t=1 means clean/fully revealed
    ts = torch.linspace(eps, 1.0 - eps, num_steps, device=device)

    for t_val in ts:
        t_batch = torch.full((batch_size,), t_val.item(), device=device)

        logits = model(x_t, t_batch) / temperature
        probs = F.softmax(logits, dim=-1)
        confidence, best_token = probs.max(dim=-1)     # (B, N) each

        # low-confidence positions get kicked back to a fresh random
        # token; confident ones get committed to the model's current guess
        low_conf = confidence < confidence_threshold
        fresh_noise = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
        x_t = torch.where(low_conf, fresh_noise, best_token)

    # final pass: commit whatever's left to the model's best guess, no more re-noising
    t_final = torch.full((batch_size,), 1.0 - eps, device=device)
    logits = model(x_t, t_final) / temperature
    x_t = logits.argmax(dim=-1)

    return x_t


# usage, matching your TextHolder's decode:
generated_ids = generate(
    whereswaldo,
    seq_len=128,
    vocab_size=vocab_size,
    num_steps=64,
    temperature=0.8,
    confidence_threshold=0.5,
    device=device,
    batch_size=4,
)

for row in generated_ids.cpu().tolist():
    print(repr(train_dataset.decode(row)))
    print("-" * 40)

'ad thou  mesers in and weect slearss the more kile the spicire hise of the dews ther make and of the cathis wonle,\nAnd the  his '
----------------------------------------
"rl you that with n me fast to sell one to the lethere thee\nAs then sure qweat in the betthou i' the dear.\n\nAULIUS:\nI on remar s "
----------------------------------------
't thases,\nIn hat was take not you that easeret thee, and have tor and hew to and not thou withte the my thance:\nTo sarms them ca'
----------------------------------------
' the s.\n\nPOLIXONES:\nNo, my br the have, plave har you foresats:\nThem, soe contand thing with of it lid,\nWearing them, sonk that '
----------------------------------------
